# Circuit Tracer model training
Train the shared one-block TensorFlow language model and export a checkpoint that the local Circuit Tracer app can load. The notebook uses the repository model, tokenizer helpers, padding-safe loss weights, a real validation split, and the required three-file checkpoint format.

## 0. Prepare the standard Colab runtime
Use a hosted Colab GPU runtime. Do not install or downgrade TensorFlow in the notebook; the standard runtime is expected to provide the versions checked below. Upload `training-data.json` before running the data cell.

In [ ]:
import subprocess
from pathlib import Path

REPOSITORY_REVISION = '132461d'
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / '.git').is_dir() and (CURRENT_DIR / 'model.py').is_file():
    REPOSITORY_DIR = CURRENT_DIR
else:
    REPOSITORY_DIR = CURRENT_DIR / 'Residual-Visualizer'
if not (REPOSITORY_DIR / '.git').is_dir():
    subprocess.run([
        'git', 'clone',
        'https://github.com/khuongngoduc0310/Residual-Visualizer.git',
        str(REPOSITORY_DIR),
    ], check=True)
else:
    print('using existing repository checkout; expected revision:', REPOSITORY_REVISION)
if not (CURRENT_DIR / '.git').is_dir():
    subprocess.run([
        'git', '-C', str(REPOSITORY_DIR), 'checkout', REPOSITORY_REVISION,
    ], check=True)
print('repository:', REPOSITORY_DIR)

In [ ]:
import json
import re
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime
from pathlib import Path

repository_candidates = [Path.cwd(), Path.cwd() / 'Residual-Visualizer']
REPOSITORY_DIR = next((
    path for path in repository_candidates if (path / 'model.py').is_file()
), None)
if REPOSITORY_DIR is None:
    raise FileNotFoundError('Clone the Residual-Visualizer repository first.')
sys.path.insert(0, str(REPOSITORY_DIR))

import numpy as np
import tensorflow as tf
from tensorflow import keras
import h5py
import keras as standalone_keras
import google.protobuf

from checkpoint import load_checkpoint, save_checkpoint
from model import ModelConfig, build_model, compile_for_training
from preprocess import (
    build_text_vectorizer,
    pad_punctuation,
    prepare_training_batch,
    validate_vocabulary,
)

if sys.version_info[:2] != (3, 13):
    raise RuntimeError(f'Python 3.13 is required; found {sys.version.split()[0]}')
expected_versions = {
    'tensorflow': '2.20.0',
    'keras': '3.13.2',
    'numpy': '2.1.3',
    'h5py': '3.16.0',
    'protobuf': '5.29.6',
}
actual_versions = {
    'tensorflow': tf.__version__,
    'keras': standalone_keras.__version__,
    'numpy': np.__version__,
    'h5py': h5py.__version__,
    'protobuf': google.protobuf.__version__,
}
if actual_versions != expected_versions:
    raise RuntimeError(f'Unsupported Colab package versions: {actual_versions}')
print('runtime versions:', actual_versions)
repository_commit = subprocess.check_output([
    'git', '-C', str(REPOSITORY_DIR), 'rev-parse', 'HEAD',
], text=True).strip()
print('repository commit:', repository_commit)

SEED = 42
tf.keras.utils.set_random_seed(SEED)

MAX_TOKENS = 10_000
MAX_LEN = 80
EMBEDDING_DIM = 256
NUM_HEADS = 2
KEY_DIM = 128
FEED_FORWARD_DIM = 256
DROPOUT_RATE = 0.1
BATCH_SIZE = 32
EPOCHS = 5
VALIDATION_SPLIT = 0.2
assert NUM_HEADS * KEY_DIM == EMBEDDING_DIM

## 1. Load training text
Set `DATA_PATH` to a JSON file, or replace `RECORDS = None` with a Python list. JSON must contain a list of strings, records with a `text` field, or wine-review records.

In [ ]:
DATA_PATH = Path('training-data.json')  # Upload this file to Colab first.
RECORDS = None  # Example: ['first training document', 'second document']

if RECORDS is None:
    if not DATA_PATH.exists():
        raise FileNotFoundError(f'Set DATA_PATH or RECORDS first. Missing: {DATA_PATH}')
    with DATA_PATH.open(encoding='utf-8') as file:
        RECORDS = json.load(file)
if not isinstance(RECORDS, list):
    raise ValueError('Training data must be a JSON list or a Python list.')

def record_to_text(record):
    if isinstance(record, str):
        return record
    if isinstance(record, dict) and isinstance(record.get('text'), str):
        return record['text']
    wine_fields = ('country', 'province', 'variety', 'description')
    if isinstance(record, dict) and all(record.get(key) is not None for key in wine_fields):
        return 'wine review : ' + ' : '.join(str(record[key]) for key in wine_fields)
    return None

raw_texts = [
    text for record in RECORDS
    if (text := record_to_text(record))
]
if len(raw_texts) < 2:
    raise ValueError('At least two valid training texts are required.')
print(f'{len(raw_texts):,} texts loaded')

In [ ]:
text_data = [pad_punctuation(text) for text in raw_texts]
rng = np.random.default_rng(SEED)
order = rng.permutation(len(text_data))
validation_count = max(1, int(len(order) * VALIDATION_SPLIT))
validation_indices = order[:validation_count]
training_indices = order[validation_count:]
validation_texts = [text_data[index] for index in validation_indices]
training_texts = [text_data[index] for index in training_indices]
if not training_texts:
    raise ValueError('The validation split left no training examples.')
print(f'train={len(training_texts):,} | validation={len(validation_texts):,}')

## 2. Tokenizer and next-token datasets
The vocabulary is learned only from training text. Sequences are right-padded, and padding targets receive zero sample weight so they do not contribute to training or validation loss.

In [ ]:
vectorizer = build_text_vectorizer(
    max_tokens=MAX_TOKENS,
    output_sequence_length=MAX_LEN + 1,
)
vectorizer.adapt(
    tf.data.Dataset.from_tensor_slices(training_texts).batch(256)
)
vocabulary = vectorizer.get_vocabulary()
validate_vocabulary(vocabulary)
print(f'learned vocabulary: {len(vocabulary):,} tokens')

def make_dataset(texts, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(texts)
    if training:
        dataset = dataset.shuffle(
            min(len(texts), 10_000),
            seed=SEED,
            reshuffle_each_iteration=True,
        )
    return (
        dataset.batch(BATCH_SIZE)
        .map(
            lambda text: prepare_training_batch(text, vectorizer),
            num_parallel_calls=tf.data.AUTOTUNE,
        )
        .prefetch(tf.data.AUTOTUNE)
    )

training_dataset = make_dataset(training_texts, training=True)
validation_dataset = make_dataset(validation_texts)
input_batch, target_batch, weight_batch = next(iter(training_dataset))
assert input_batch.shape[1:] == (MAX_LEN,)
assert target_batch.shape == input_batch.shape
assert weight_batch.shape == target_batch.shape
_, verification_targets, verification_weights = prepare_training_batch(
    tf.constant(['verification sample']),
    vectorizer,
)
assert tf.reduce_any(verification_targets != 0)
assert tf.reduce_any(verification_targets == 0)
assert tf.reduce_all(
    tf.boolean_mask(verification_weights, verification_targets == 0) == 0
)
assert tf.reduce_all(
    tf.boolean_mask(verification_weights, verification_targets != 0) == 1
)
print('inputs/targets/weights:', input_batch.shape, target_batch.shape, weight_batch.shape)

## 3. Shared model
The shared post-norm transformer uses a causal mask. Because padding is on the right, real tokens cannot attend to later padding; zero sample weights remove padded targets from the loss.

In [ ]:
config = ModelConfig(
    vocab_size=len(vocabulary),
    max_len=MAX_LEN,
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    key_dim=KEY_DIM,
    feed_forward_dim=FEED_FORWARD_DIM,
    dropout_rate=DROPOUT_RATE,
)
language_model = build_model(config)
compile_for_training(language_model)

test_inputs = input_batch[:2]
test_probabilities = language_model(test_inputs, training=False)
assert test_probabilities.shape == (
    test_inputs.shape[0], MAX_LEN, len(vocabulary)
)
np.testing.assert_allclose(
    tf.reduce_sum(test_probabilities, axis=-1).numpy(),
    1.0,
    rtol=1e-5,
    atol=1e-6,
)
language_model.summary()

## 4. Train
Temporary training weights are separate from the final app checkpoint. Early stopping restores the model state with the best validation loss.

In [ ]:
TRAINING_ARTIFACT_DIR = REPOSITORY_DIR / 'training-artifacts'
TRAINING_ARTIFACT_DIR.mkdir(exist_ok=True)
callbacks = [
    keras.callbacks.ModelCheckpoint(
        TRAINING_ARTIFACT_DIR / 'best.weights.h5',
        save_weights_only=True,
        save_best_only=True,
        monitor='val_loss',
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        patience=1,
        factor=0.5,
    ),
]
history = language_model.fit(
    training_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## 5. Export and verify the app checkpoint
Export into a new folder, then reload it and compare predictions before downloading it.

In [ ]:
EXPORT_ROOT = REPOSITORY_DIR / 'exports'
EXPORT_ROOT.mkdir(exist_ok=True)
CHECKPOINT_DIR = EXPORT_ROOT / datetime.now().strftime('checkpoint-%Y%m%d-%H%M%S')

expected_predictions = language_model(input_batch[:1], training=False).numpy()
save_checkpoint(
    CHECKPOINT_DIR,
    language_model,
    vocabulary,
    config,
)
loaded_checkpoint = load_checkpoint(CHECKPOINT_DIR)
actual_predictions = loaded_checkpoint.model(
    input_batch[:1],
    training=False,
).numpy()
np.testing.assert_allclose(
    expected_predictions,
    actual_predictions,
    rtol=1e-6,
    atol=1e-7,
)
expected_files = {'model.weights.h5', 'vocabulary.json', 'config.json'}
assert {path.name for path in CHECKPOINT_DIR.iterdir()} == expected_files
print('verified checkpoint:', CHECKPOINT_DIR)

In [ ]:
archive_path = Path(shutil.make_archive(
    str(CHECKPOINT_DIR),
    'zip',
    root_dir=CHECKPOINT_DIR.parent,
    base_dir=CHECKPOINT_DIR.name,
))
with zipfile.ZipFile(archive_path) as checkpoint_archive:
    archived_files = {
        Path(name).name for name in checkpoint_archive.namelist() if not name.endswith('/')
    }
assert archived_files == expected_files
print('checkpoint archive:', archive_path)
try:
    from google.colab import files
    files.download(str(archive_path))
except (ImportError, RuntimeError):
    print('Download is unavailable; copy the archive from the path above.')

## 6. Generate text
Generation samples from the model's softmax probabilities and never exceeds the 80-position embedding limit.

In [ ]:
def generate(prompt, max_new_tokens=40, temperature=0.8, top_k=20):
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError('Prompt must contain text.')
    if max_new_tokens < 0:
        raise ValueError('max_new_tokens must not be negative.')
    if temperature <= 0:
        raise ValueError('temperature must be positive.')
    if isinstance(top_k, bool) or not isinstance(top_k, int) or top_k <= 0:
        raise ValueError('top_k must be a positive integer.')

    token_ids = vectorizer(tf.constant([pad_punctuation(prompt)]))[0]
    token_ids = tf.boolean_mask(token_ids, token_ids != 0)
    token_count = int(tf.size(token_ids))
    if token_count == 0:
        raise ValueError('Prompt does not contain any tokens.')
    if token_count > MAX_LEN:
        raise ValueError(f'Prompt has {token_count} tokens; maximum is {MAX_LEN}.')

    target_length = min(MAX_LEN, token_count + max_new_tokens)
    while int(tf.size(token_ids)) < target_length:
        probabilities = language_model(
            token_ids[None, :],
            training=False,
        )[0, -1, :]
        values, indices = tf.math.top_k(
            probabilities,
            k=min(top_k, len(vocabulary)),
        )
        sampling_logits = tf.math.log(tf.clip_by_value(values, 1e-9, 1.0))
        sampled_index = tf.random.categorical(
            sampling_logits[None, :] / temperature,
            1,
        )[0, 0]
        next_token_id = int(indices[int(sampled_index.numpy())].numpy())
        if next_token_id == 0:
            break
        token_ids = tf.concat([token_ids, [next_token_id]], axis=0)

    text = ' '.join(vocabulary[token_id] for token_id in token_ids.numpy())
    return re.sub(r'\s+([.,!?;:])', r'\1', text)

print(generate('wine review : us', temperature=0.8))

## 7. Basic residual norms
This training notebook plots only the embedding and final block output from the shared model. Deeper internal locations belong to the app's dedicated inspection model.

In [ ]:
embedding_layer = language_model.get_layer('token_and_position_embedding')
transformer_layer = language_model.get_layer('transformer_block')
residual_probe = keras.Model(
    language_model.input,
    [embedding_layer.output, transformer_layer.output[0]],
)

def plot_residual_norms(text):
    token_ids = vectorizer(tf.constant([pad_punctuation(text)]))[0]
    token_ids = tf.boolean_mask(token_ids, token_ids != 0)
    token_count = int(tf.size(token_ids))
    if token_count == 0:
        raise ValueError('Text does not contain any tokens.')
    if token_count > MAX_LEN:
        raise ValueError(f'Text has {token_count} tokens; maximum is {MAX_LEN}.')

    embedding_output, block_output = residual_probe(
        token_ids[None, :],
        training=False,
    )
    tokens = [vocabulary[token_id] for token_id in token_ids.numpy()]
    plt.figure(figsize=(max(8, len(tokens) * 0.45), 4))
    for residual, label in [
        (embedding_output, 'embedding'),
        (block_output, 'block output'),
    ]:
        plt.plot(
            tf.norm(residual[0], axis=-1).numpy(),
            marker='o',
            label=label,
        )
    plt.xticks(range(len(tokens)), tokens, rotation=60, ha='right')
    plt.ylabel(r'$\|r_t\|_2$')
    plt.xlabel('token')
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_residual_norms('wine review : us : california : pinot noir')